# Neural Networks | Solution Notebook

This notebook contains worked solutions for all labs in Module 3. Use it after completing the live labs to compare your approach, check your reasoning, and review the judgement calls behind each decision.

**Important:** There is no single correct answer for most tasks. These solutions represent strong answer shapes. Your approach may differ and still be valid if it is well-justified.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (8, 5)

runs = pd.read_csv(DATA_DIR / "training_runs.csv")
labels = pd.read_csv(DATA_DIR / "document_classification_labels.csv")

print(f"Training runs: {runs.shape}")
print(f"Document labels: {labels.shape}")

## Lab 1 Solution | Network Architecture Exploration

**Architecture sketch description:**
- Input layer: one neuron per feature (e.g. 10 features = 10 input neurons).
- Hidden layer 1: 64 neurons with ReLU activation.
- Hidden layer 2: 32 neurons with ReLU activation.
- Output layer: 1 neuron with sigmoid activation (binary classification).

**Plain-language explanations:**
- **Weights:** Each connection between neurons has a weight that determines how much influence one neuron has on the next. Larger weights mean stronger influence. The network learns by adjusting these weights.
- **Bias:** A bias term allows each neuron to shift its output independently of inputs. Think of it as a baseline or offset that helps the neuron activate appropriately.
- **Activation function:** After computing a weighted sum, the activation function decides how strongly the neuron "fires." ReLU passes positive values through and blocks negative values. Sigmoid squashes output between 0 and 1, useful for probability-like outputs.
- **Loss function:** Measures how wrong the network's predictions are compared to the true answers. The network's goal is to minimise this value. Binary cross-entropy measures how far predicted probabilities are from actual 0/1 labels.
- **Backpropagation:** After making a prediction and measuring the error, the network works backward through its layers to calculate how much each weight contributed to the error, then adjusts the weights to reduce the error next time.
- **Learning rate:** Controls how large each weight adjustment is. Too high and the network overshoots good solutions; too low and it learns too slowly or gets stuck.

**Likely failure mode:** Overfitting -- the network memorises training examples instead of learning general patterns, performing well on training data but poorly on new data.

In [ ]:
# Lab 2 Solution | Data Preparation

print("Training runs data overview:")
print(f"Shape: {runs.shape}")
print(f"\nData types:\n{runs.dtypes}")
print(f"\nMissing values:\n{runs.isnull().sum()}")
print(f"\nBasic statistics:")
print(runs.describe())

# Data preparation principles for neural networks:
#
# 1. Scaling is essential. Neural networks use gradient-based optimisation,
#    which is sensitive to feature magnitudes. StandardScaler (mean=0, std=1)
#    or MinMaxScaler (0-1 range) are appropriate choices.
#
# 2. Categorical features must be encoded. One-hot encoding is standard
#    for nominal categories. Label encoding can work for ordinal categories.
#
# 3. Train/validation/test split should be done before scaling to prevent
#    leaking validation/test statistics into the training set. Fit the scaler
#    on training data only, then transform validation and test sets.
#
# 4. Class distribution should be checked across all splits. If imbalanced,
#    consider stratified splitting, class weights, or oversampling.
#
# 5. Why neural networks need scaling but trees do not: decision trees split
#    on individual features independently, so scale does not matter. Neural
#    networks combine features through weighted sums, so a feature with values
#    in the millions will dominate features with values near zero.

print("\nKey insight: scale after splitting, using training statistics only.")

In [ ]:
# Lab 3 Solution | Basic Model Training

# Visualise training runs to demonstrate curve reading
print("Available columns:", list(runs.columns))

# Example training curve analysis
# Adapt column names to match the actual dataset

# Pattern for interpreting training curves:
#
# 1. Both train and val loss decreasing: model is learning, continue training.
# 2. Train loss decreasing, val loss increasing: overfitting.
#    Action: add regularisation (dropout, weight decay), reduce model size,
#    or stop training earlier.
# 3. Both losses flat from the start: model is not learning.
#    Action: check learning rate (too low?), check data preparation,
#    check architecture (too simple for the task?).
# 4. Both losses decreasing but slowly: model is learning but may need
#    more epochs or a higher learning rate.
# 5. Loss is noisy (jumping up and down): batch size may be too small
#    or learning rate too high.

# Diagnosis template:
# "Validation loss begins to increase after epoch [X] while training loss
# continues to decrease, indicating overfitting. The gap between training
# and validation loss at epoch [X] is [Y], suggesting the model has begun
# to memorise training examples. Recommended action: apply early stopping
# at epoch [X] and consider adding dropout regularisation."

print("Key insight: training curves tell you what is happening and what to do next.")
print("A flat curve is not 'the model has learned' -- it may mean 'the model has stalled.'")

## Lab 4 Solution | CNN Concepts for Document Understanding

**Convolution explained practically:**
A small filter (e.g. 3x3 pixels) slides across the document image, detecting local patterns at each position. Early layers detect simple patterns like edges, lines, and corners. Deeper layers combine these into higher-level features like text blocks, logos, or layout structures. This is useful for documents because the same patterns (e.g. a header format, a signature region) appear at different positions in different documents.

**Pooling explained practically:**
After convolution, pooling reduces the spatial size of the feature maps by keeping only the most important information from small regions. Max pooling, for example, keeps the strongest activation in each region. This makes the network more robust to small shifts in position and reduces computation.

**Transfer learning recommendation:**
Use transfer learning. Training a CNN from scratch requires tens of thousands of labelled document images, which most banks do not have. A pre-trained model (e.g. ResNet-50 trained on ImageNet) already knows general visual features (edges, textures, shapes). Fine-tuning the top layers on a few hundred to a few thousand labelled banking documents is more practical and likely to produce better results.

**Domain gap assessment:**
ImageNet contains natural photographs (animals, objects, scenes). Banking documents are structured, text-heavy, and have consistent layouts. The domain gap is moderate: low-level features (edges, lines) transfer well, but high-level features (object recognition) are less relevant. Focus fine-tuning on the classification head and top convolutional layers.

**Key data quality risk:**
Scan quality variation. Banking documents come from scanners, phone cameras, and digital submissions with different resolutions, lighting, and orientations. A model trained on clean scans may fail on lower-quality inputs.

In [ ]:
# Lab 5 Solution | Transfer Learning Application

print("Document classification label distribution:")
print(labels["document_type"].value_counts())

# Transfer learning strategy:
#
# Pre-trained model: EfficientNet-B0
# Justification: Good balance of accuracy and computational cost. Smaller
# than ResNet-50 but competitive performance. Suitable for a bank that
# may have limited GPU resources.
#
# Layer freezing strategy:
# Phase 1: Freeze all convolutional layers. Train only the classification
#   head (new fully connected layers) for 5-10 epochs.
# Phase 2: Unfreeze the top 2-3 convolutional blocks. Fine-tune with a
#   low learning rate (1e-5) for 10-20 epochs.
# Phase 3 (optional): Unfreeze all layers with a very low learning rate
#   (1e-6) if validation accuracy is still improving.
#
# Data requirements:
# Minimum: 200-500 labelled examples per document type for basic fine-tuning.
# Recommended: 500-2000 per type for robust performance.
# These numbers are realistic for a bank with an existing document archive.
#
# Negative transfer risk:
# If the pre-trained features are too different from banking documents,
# fine-tuning may produce worse results than training a smaller model from
# scratch. Detect this by comparing fine-tuned validation accuracy against
# a simple baseline (e.g. metadata-only classifier).

print("\nKey insight: validate that transfer learning actually helps.")
print("Always compare against a simpler baseline.")

In [ ]:
# Lab 6 Solution | Network Tuning and Performance

# Example comparison of two training runs
# Adapt to actual column names and run IDs

# Comparison pattern:
# "Run A reaches validation loss of 0.34 at epoch 20 and then plateaus.
#  Run B reaches 0.31 at epoch 25 but shows an increasing gap between
#  training and validation loss, suggesting early overfitting.
#
#  Recommendation: Run A. Although Run B achieves lower final validation
#  loss, its overfitting trend suggests it will not generalise reliably
#  to new data. Run A's stable validation curve indicates more predictable
#  production performance.
#
#  Caveats:
#  1. If Run B's overfitting is addressed with regularisation, it may
#     become the better choice.
#  2. Neither run has been tested on a held-out test set representative
#     of production data.
#  3. Performance on edge cases (poor scan quality, unusual document
#     types) has not been assessed."

monitoring = pd.DataFrame({
    "Metric": ["Weekly Accuracy", "Confidence Distribution", "Input Drift (PSI)"],
    "Threshold": ["< 0.80", "Mean shift > 0.10", "> 0.15"],
    "Response": ["Investigate; hold new deployments",
                 "Review recent documents; check for new types",
                 "Flag for retraining review"],
    "Owner": ["Model Owner", "Model Owner", "Model Risk"]
})
print("Monitoring checklist:")
print(monitoring.to_string(index=False))

print("\nRollback trigger: rollback if weekly accuracy drops below 0.75")
print("for two consecutive weeks, or immediately if a critical misclassification")
print("is reported (e.g. identity document classified as correspondence).")

## Capstone Solution Shape | Recommend or Reject the Neural Path

A strong capstone recommendation will include:

**1. Business Problem:** Automated classification of incoming banking documents (loan applications, identity documents, correspondence, statements) to route them to the correct processing team. Current manual routing is slow and error-prone.

**2. Representation Challenge:** Documents are image-based with variable scan quality, layouts, and formats. The classification task depends on visual structure, not just text content, which motivates a neural approach.

**3. Recommendation:** Adopt a CNN with transfer learning (EfficientNet-B0) for a pilot. The visual structure of banking documents is complex enough to benefit from convolutional feature extraction, and transfer learning reduces the data and compute requirements to a level realistic for AJB.

**4. Alternative Considered:** A metadata-only classifier (document size, submission channel, customer type) using a random forest. This is simpler and more interpretable but cannot use the visual content of documents, limiting accuracy on ambiguous cases.

**5. Governance Burden:**
- CNNs are harder to explain than tree-based models. Compliance review will require visual explanations (e.g. GradCAM highlighting which regions influenced the decision).
- Retraining requires GPU access and takes longer than retraining a simpler model.
- New document types require labelled examples and model revalidation.

**6. Monitoring and Controls:**
- Weekly accuracy and confidence monitoring with named owners.
- Rollback trigger at sustained accuracy below 0.75.
- Human review for predictions below 0.80 confidence.
- Quarterly governance review with Model Risk.

**7. Risk Summary:**
- Scan quality variation may degrade performance on low-quality inputs (mitigated by including varied quality in training data).
- New document types not seen in training will be misclassified (mitigated by confidence thresholds and human review).
- Explainability burden is higher than for simpler models (mitigated by GradCAM and clear documentation).

**8. Next Step:** Pilot on one document type (loan applications) for 6 weeks, processing alongside manual classification to measure accuracy, speed, and user satisfaction. Scale to additional document types only after pilot success criteria are met (accuracy > 0.85, no critical misclassifications, processing team endorsement).